## Quantum Chess Engine

Luis Baeza, Joshua Longoria

---

This notebook implements and demonstrates the quantum mechanics behind **Quantum Chess**.
Three quantum moves extend the classical game:

| Move | Gate(s) | Effect |
|---|---|---|
| Superposition | H | Piece exists on two squares until observed |
| Entanglement | H + CNOT | Two pieces collapse together (Bell state) |
| Measurement | Measure | Forces a superposed piece to pick one square |

Each section builds the Qiskit circuit, visualizes it, and runs a simulation.

In [1]:
# Cell name: Imports
# Purpose: Load all required Qiskit and visualization libraries
# References: Qiskit Documentation — https://docs.quantum.ibm.com/
#             Course lecture: 'My first qiskit program - Build Quantum Circuit'
# Contributor: Luis Baeza | 04/21/2026 — Initial implementation

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.providers.basic_provider import BasicSimulator
from qiskit import transpile
from qiskit.visualization import plot_histogram
import random

simulator = BasicSimulator()
print("Qiskit imports OK — simulator ready")

Qiskit imports OK — simulator ready


---
## Superposition (Hadamard Gate)

When a player makes a **superposition move**, their piece splits across two target squares.
This is modeled by applying a **Hadamard gate** to a single qubit:

$$H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}$$

- Outcome `0` → piece is at **square A**
- Outcome `1` → piece is at **square B**

Both outcomes have equal probability (50/50) until the piece is measured.

In [ ]:
# Cell name: Superposition Circuit — Build and Draw
# Purpose: Build a 1-qubit Hadamard circuit representing a piece in superposition
# References: Qiskit QuantumCircuit.h() documentation
#             Course lecture: 'My first qiskit program' — Cell 6 (H gate)
# Contributor: Luis Baeza | 04/21/2026 — Initial implementation

qr_super = QuantumRegister(1, 'piece')
cr_super = ClassicalRegister(1, 'square')
qc_super = QuantumCircuit(qr_super, cr_super)

qc_super.h(qr_super[0])                       # H gate — puts piece into superposition
qc_super.measure(qr_super[0], cr_super[0])    # measure — collapses to square A (0) or B (1)

qc_super.draw('mpl')

In [ ]:
# Cell name: Superposition Circuit — Simulate 1000 Shots
# Purpose: Show that the H gate produces a 50/50 distribution over many runs
# References: Qiskit BasicSimulator, transpile documentation
#             Course lecture: 'My first qiskit program' — Cell 19 (transpile + run)
# Contributor: Luis Baeza | 04/21/2026 — Initial implementation

compiled_super = transpile(qc_super, simulator)
job_super      = simulator.run(compiled_super, shots=1000)
counts_super   = job_super.result().get_counts()

print("Superposition outcomes (1000 shots):")
print(f"  Square A (|0>): {counts_super.get('0', 0)} times")
print(f"  Square B (|1>): {counts_super.get('1', 0)} times")

plot_histogram(counts_super)

---
## Entanglement (Bell State via H + CNOT)

When a player makes an **entangle move**, two pieces become correlated.
This is modeled by creating a **Bell state** using H on the control qubit followed by a CNOT:

$$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$$

The key property: **both qubits always collapse to the same value.**
- If piece A collapses to index `0` (its square A), piece B also collapses to index `0`
- If piece A collapses to index `1` (its square B), piece B also collapses to index `1`

Measuring one piece instantly determines the other — no matter where they are on the board.

In [ ]:
# Cell name: Bell State Circuit — Build and Draw
# Purpose: Build the 2-qubit entanglement circuit (H + CNOT = Bell state)
# References: Qiskit QuantumCircuit.cx() documentation
#             Course lecture: 'Qiskit_Overview' — Bell state cells
#             Course lecture: 'My first qiskit program' — Cell 11 (CNOT)
# Contributor: Luis Baeza | 04/21/2026 — Initial implementation

qr_bell = QuantumRegister(2, 'pieces')
cr_bell = ClassicalRegister(2, 'squares')
qc_bell = QuantumCircuit(qr_bell, cr_bell)

qc_bell.h(qr_bell[0])                # H on control — creates superposition
qc_bell.cx(qr_bell[0], qr_bell[1])  # CNOT — entangles control with target
qc_bell.measure(qr_bell, cr_bell)

qc_bell.draw('mpl')

In [ ]:
# Cell name: Bell State Circuit — Simulate 1000 Shots
# Purpose: Confirm that entangled pieces always collapse together (only '00' and '11')
# References: Qiskit BasicSimulator, plot_histogram
#             Course lecture: 'Qiskit_Overview' — Bell state simulation
# Contributor: Luis Baeza | 04/21/2026 — Initial implementation

compiled_bell = transpile(qc_bell, simulator)
job_bell      = simulator.run(compiled_bell, shots=1000)
counts_bell   = job_bell.result().get_counts()

print("Bell state outcomes (1000 shots):")
for outcome, count in sorted(counts_bell.items()):
    print(f"  |{outcome}> : {count} times")
print("\nOnly '00' and '11' appear — entangled pieces always collapse together.")

plot_histogram(counts_bell)

---
## Single-Shot Measurement

In the actual chess game, each quantum event runs the circuit **once** (`shots=1`).
This gives a single probabilistic outcome — the piece collapses to one definite square.

The two helper functions below show what a single measurement looks like
for both the superposition and entanglement cases.

In [2]:
# Cell name: Single-Shot Measurement Functions
# Purpose: Demonstrate single-shot circuit runs that produce one chess outcome
# References: Qiskit BasicSimulator shots parameter
# Contributor: Luis Baeza | 04/21/2026 — Initial implementation

def measure_superposed_piece(sq_a: str, sq_b: str) -> str:
    """Run superposition circuit once. Returns the square the piece collapses to."""
    qr = QuantumRegister(1, 'q')
    cr = ClassicalRegister(1, 'c')
    qc = QuantumCircuit(qr, cr)
    qc.h(qr[0])
    qc.measure(qr[0], cr[0])

    bits   = list(simulator.run(transpile(qc, simulator), shots=1).result().get_counts().keys())[0]
    result = int(bits)
    return sq_a if result == 0 else sq_b


def measure_entangled_pieces(sq_a1: str, sq_a2: str,
                              sq_b1: str, sq_b2: str) -> tuple[str, str]:
    """
    Run Bell state circuit once.
    Returns (square_for_piece_A, square_for_piece_B) — always correlated.
    Note: Qiskit returns bits in LSB-first order, so bits[-1] is qubit 0.
    """
    qr = QuantumRegister(2, 'q')
    cr = ClassicalRegister(2, 'c')
    qc = QuantumCircuit(qr, cr)
    qc.h(qr[0])
    qc.cx(qr[0], qr[1])
    qc.measure(qr, cr)

    bits = list(simulator.run(transpile(qc, simulator), shots=1).result().get_counts().keys())[0]
    r0 = int(bits[-1])   # qubit 0 result
    r1 = int(bits[-2])   # qubit 1 result
    return (sq_a1 if r0 == 0 else sq_a2), (sq_b1 if r1 == 0 else sq_b2)

In [ ]:
# Cell name: Single-Shot Demonstration
# Purpose: Run each single-shot function and show chess-context output
# References: —
# Contributor: Luis Baeza | 04/21/2026 — Initial implementation

# Superposition: knight split between e4 and g4
print("--- Superposition demo ---")
for _ in range(5):
    sq = measure_superposed_piece('e4', 'g4')
    print(f"  Knight collapsed to: {sq}")

print()

# Entanglement: rook A between a1/a3, rook B between h1/h3
print("--- Entanglement demo ---")
for _ in range(5):
    sq_a, sq_b = measure_entangled_pieces('a1', 'a3', 'h1', 'h3')
    print(f"  Rook A -> {sq_a}  |  Rook B -> {sq_b}  (always same index)")

---
## QuantumChessEngine Class

The `QuantumChessEngine` class packages all the above circuits into a single object
that `quantum_rules.py` calls during a live game.

It tracks which qubits are **superposed** or **entangled**, then builds and runs
the appropriate Qiskit circuit when `measure()` is called.

In [3]:
# Cell name: QuantumChessEngine Class
# Purpose: Encapsulate all quantum operations used by the chess game
# References: Qiskit QuantumCircuit, BasicSimulator, transpile
#             Section 1–3 circuits above
# Contributor: Luis Baeza | 04/21/2026 — Initial implementation

class QuantumChessEngine:
    """
    Qiskit-based quantum engine for Quantum Chess.

    Tracks qubit states and runs single-shot Qiskit circuits on BasicSimulator
    when a piece is measured. Interface matches the game-compatible
    QuantumEngine class in Quantum_engin.py.

    States:
        classical  — piece is at one definite square (qubit in |0>)
        superposed — piece split across two squares (H gate applied)
        entangled  — piece linked to a partner (Bell state)
    """

    def __init__(self):
        self.simulator   = BasicSimulator()
        self._superposed = set()   # qubit IDs currently in superposition
        self._entangled  = {}      # qubit_id -> partner_qubit_id

    # ------------------------------------------------------------------
    # Gate operations
    # ------------------------------------------------------------------

    def apply_hadamard(self, qubit_id: int):
        """Apply H gate: mark qubit as superposed."""
        self._superposed.add(qubit_id)

    def apply_cnot(self, control_id: int, target_id: int):
        """Apply CNOT gate: entangle control and target into a Bell state."""
        self._entangled[control_id] = target_id
        self._entangled[target_id]  = control_id

    # ------------------------------------------------------------------
    # Measurement
    # ------------------------------------------------------------------

    def measure(self, qubit_id: int) -> tuple:
        """
        Measure a qubit by running a Qiskit circuit on BasicSimulator.

        Returns:
            (result, partner_result)
            result         — 0 or 1 (index into the piece's positions list)
            partner_result — same value if entangled, None otherwise
        """
        if qubit_id in self._entangled:
            return self._measure_entangled(qubit_id)

        if qubit_id not in self._superposed:
            return 0, None   # classical piece always at positions[0]

        return self._measure_superposed(qubit_id)

    def _measure_superposed(self, qubit_id: int) -> tuple:
        """Run a 1-qubit H+measure circuit. Returns (0|1, None)."""
        qr = QuantumRegister(1, 'q')
        cr = ClassicalRegister(1, 'c')
        qc = QuantumCircuit(qr, cr)
        qc.h(qr[0])
        qc.measure(qr[0], cr[0])

        bits   = list(self.simulator.run(transpile(qc, self.simulator), shots=1)
                      .result().get_counts().keys())[0]
        result = int(bits)

        self._superposed.discard(qubit_id)
        return result, None

    def _measure_entangled(self, qubit_id: int) -> tuple:
        """Run a 2-qubit Bell state circuit. Returns (r0, r1) — always equal."""
        partner_id = self._entangled[qubit_id]

        qr = QuantumRegister(2, 'q')
        cr = ClassicalRegister(2, 'c')
        qc = QuantumCircuit(qr, cr)
        qc.h(qr[0])
        qc.cx(qr[0], qr[1])
        qc.measure(qr, cr)

        bits = list(self.simulator.run(transpile(qc, self.simulator), shots=1)
                    .result().get_counts().keys())[0]
        r0 = int(bits[-1])   # qubit 0 — LSB first in Qiskit
        r1 = int(bits[-2])   # qubit 1

        # Collapse both qubits
        self._superposed.discard(qubit_id)
        self._superposed.discard(partner_id)
        del self._entangled[qubit_id]
        if partner_id in self._entangled:
            del self._entangled[partner_id]

        return r0, r1

    # ------------------------------------------------------------------
    # State queries
    # ------------------------------------------------------------------

    def is_superposed(self, qubit_id: int) -> bool:
        return qubit_id in self._superposed

    def is_entangled(self, qubit_id: int) -> bool:
        return qubit_id in self._entangled

    def get_partner(self, qubit_id: int):
        """Return the partner qubit ID if entangled, else None."""
        return self._entangled.get(qubit_id)


print("QuantumChessEngine class defined.")

QuantumChessEngine class defined.


In [4]:
# Cell name: Chess Scenario Demo
# Purpose: Show the engine working in a realistic chess context
# References: —
# Contributor: Luis Baeza | 04/21/2026 — Initial implementation

engine = QuantumChessEngine()

# --- Scenario 1: Superposition move ---
print("=== Scenario 1: Superposition ===")
knight_qubit = 10
engine.apply_hadamard(knight_qubit)
print(f"Knight in superposition: {engine.is_superposed(knight_qubit)}")

# Simulate measuring the knight 5 times (new engine each time)
positions = []
for _ in range(5):
    e = QuantumChessEngine()
    e.apply_hadamard(0)
    result, _ = e.measure(0)
    positions.append('e4' if result == 0 else 'g4')
print(f"5 collapse outcomes: {positions}")

print()

# --- Scenario 2: Entanglement ---
print("=== Scenario 2: Entanglement ===")
rook_a, rook_b = 0, 7
engine2 = QuantumChessEngine()
engine2.apply_hadamard(rook_a)
engine2.apply_cnot(rook_a, rook_b)
print(f"Rook A entangled: {engine2.is_entangled(rook_a)}")
print(f"Rook B entangled: {engine2.is_entangled(rook_b)}")

pairs = []
for _ in range(5):
    e = QuantumChessEngine()
    e.apply_hadamard(0)
    e.apply_cnot(0, 1)
    r0, r1 = e.measure(0)
    pairs.append((r0, r1))
print("5 entangled collapses (r_A, r_B) — should always match:")
for r_a, r_b in pairs:
    match = 'OK' if r_a == r_b else 'MISMATCH'
    print(f"  ({r_a}, {r_b}) {match}")

=== Scenario 1: Superposition ===
Knight in superposition: True
5 collapse outcomes: ['g4', 'e4', 'e4', 'e4', 'g4']

=== Scenario 2: Entanglement ===
Rook A entangled: True
Rook B entangled: True
5 entangled collapses (r_A, r_B) — should always match:
  (0, 0) OK
  (0, 0) OK
  (0, 0) OK
  (0, 0) OK
  (0, 0) OK


---
## Summary

| Component | Circuit | Result |
|---|---|---|
| Superposition | 1-qubit H + measure | 50/50 collapse to square A or B |
| Entanglement | 2-qubit H + CNOT + measure | Both pieces always collapse to same index |
| Classical | No circuit | Always returns 0 (piece stays put) |

The `QuantumChessEngine` class above is the **Qiskit reference implementation**.
The file `Quantum_engin.py` contains a game-compatible version with the same interface
that uses Python `random` so the chess game can run without requiring the cs3368 environment.